# Iris-3B 全体FT · 新しいデータで継続学習

この公開版で保存した全体FTチェックポイントから、重み・optimizer・scheduler・乱数を引き継ぎます。
新しいデータの位置は0から始め、optimizer更新数は通算で数えます。
先に親runを停止し、同時実行を避けてください。A100 80GB・高RAM・十分なDrive容量が必要です。

`PARENT_RUN_NAME`、別名の`RUN_NAME`、データセットと列名を設定します。seed・解像度・バッチ・warmup等は親と一致させます。
`MAX_STEPS`は通算の目標です。学習率を変える場合は、親の設定値を`PARENT_LEARNING_RATE`に明示します。
以前の画像を除外する場合は、Drive内のdataset manifestの相対パスを`EXCLUSION_MANIFESTS_JSON`にJSON配列で指定します。
除外には画像IDとSHA-256を使うため、画像IDはデータセット間でも一意になるよう用意してください。


In [ ]:
#@title 依存関係と固定revisionのIrisソースを準備
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
import importlib.metadata as metadata
import subprocess
import sys
from pathlib import Path
from packaging.version import Version

if sys.version_info < (3, 11):
    raise RuntimeError("Python 3.11以上のColabランタイムを選んでください。")
pins = {
    "bitsandbytes": ("0.49.2", "bitsandbytes"),
    "transformers": ("5.10.4", "transformers"),
    "huggingface-hub": ("1.5.0", "huggingface_hub"),
    "safetensors": ("0.8.0", "safetensors"),
    "omegaconf": ("2.3.1", "omegaconf"),
    "pillow": ("12.3.0", "PIL"),
    "numpy": ("2.3.5", "numpy"),
    "pyarrow": ("26.0.0", "pyarrow"),
}
requirements, restart = [], []
for distribution, (wanted, module) in pins.items():
    try:
        installed = metadata.version(distribution)
    except metadata.PackageNotFoundError:
        installed = None
    if installed != wanted:
        requirements.append(f"{distribution}=={wanted}")
        if module in sys.modules:
            restart.append(module)
try:
    torch_version = metadata.version("torch")
except metadata.PackageNotFoundError:
    torch_version = "0"
if Version(torch_version.split("+")[0]) < Version("2.7.1"):
    requirements.append("torch>=2.7.1,<2.14")
    if "torch" in sys.modules:
        restart.append("torch")
if requirements:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                    "--disable-pip-version-check", *requirements], check=True)
if restart:
    raise RuntimeError("依存関係を更新しました。ランタイムのセッションを再起動して、上から再実行してください。")

SOURCE_REVISION = "a8d15239dea469aba042cfa56ca3bb4e450d5ebc"
SOURCE_ROOT = Path("/content/iris-3b-source")
if not SOURCE_ROOT.exists():
    subprocess.run(["git", "clone", "--quiet", "--no-checkout",
                    "https://github.com/speridlabs/iris-3b.git", str(SOURCE_ROOT)], check=True)
    subprocess.run(["git", "-C", str(SOURCE_ROOT), "checkout", "--quiet", "--detach", SOURCE_REVISION], check=True)
actual = subprocess.check_output(["git", "-C", str(SOURCE_ROOT), "rev-parse", "HEAD"], text=True).strip()
if actual != SOURCE_REVISION:
    raise RuntimeError("Irisソースのrevisionが違います。新しいランタイムで上から実行してください。")
sys.path.insert(0, str(SOURCE_ROOT / "src"))
print("環境を準備しました。Iris revision:", actual[:12])


In [ ]:
%%writefile /content/iris_lora_colab.py
"""Iris-3B LoRA training, fixed data selection, and resumable Drive checkpoints.

The notebook embeds this file verbatim. The original Iris architecture, text
encoder, rectified-flow loss, and sampler are imported from the pinned upstream.
"""
from __future__ import annotations

import copy
import errno
import fnmatch
import gc
import hashlib
import io
import json
import math
import os
import pickle
import random
import re
import shutil
import tarfile
import time
import uuid
from dataclasses import asdict
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image, ImageOps, features
from torch import nn

SOURCE_REVISION = "a8d15239dea469aba042cfa56ca3bb4e450d5ebc"
BASE_REPO = "speridlabs/iris-3b"
BASE_REVISION = "7445443349bc9abe3c96f01ff793e2098ca012b3"
TEXT_REPO = "Qwen/Qwen3-VL-4B-Instruct"
TEXT_REVISION = "ebb281ec70b05090aa6165b016eac8ec08e71b17"
DEFAULT_DATA_COLUMNS = {
    "image_id": "id", "image": "image", "prompt": "prompt", "aesthetic_tier": None,
}
SCHEMA_VERSION = 2
IMAGE_NAME = re.compile(r"images/[0-9]{6}\.bin\Z")
CHECKPOINT_NAME = re.compile(r"step-[0-9]{8}-[a-f0-9]{8}\.pt\Z")


def stable_hash(value):
    return hashlib.sha256(json.dumps(value, ensure_ascii=False, sort_keys=True,
                                     separators=(",", ":")).encode("utf-8")).hexdigest()


def file_hash(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def flush_file(stream):
    stream.flush()
    try:
        os.fsync(stream.fileno())
    except OSError as exc:
        # Drive/FUSE implementations may not expose fsync. Readback checks
        # still apply; real I/O and quota failures must remain errors.
        if exc.errno not in {errno.EINVAL, errno.ENOSYS, errno.ENOTSUP, errno.EOPNOTSUPP}:
            raise


def atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_name(path.name + "." + uuid.uuid4().hex + ".partial")
    try:
        serialized = json.dumps(value, ensure_ascii=False, indent=2)
        with temp.open("w", encoding="utf-8", newline="\n") as stream:
            stream.write(serialized)
            flush_file(stream)
        if file_hash(temp) != hashlib.sha256(serialized.encode("utf-8")).hexdigest():
            raise OSError("JSON保存のSHA-256が一致しません。")
        os.replace(temp, path)
    finally:
        temp.unlink(missing_ok=True)


def verified_copy(source, destination, expected_hash=None):
    """Publish the destination only after a complete copy and checksum readback."""
    source, destination = Path(source), Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    checksum = expected_hash or file_hash(source)
    temp = destination.with_name(destination.name + "." + uuid.uuid4().hex + ".partial")
    try:
        with source.open("rb") as incoming, temp.open("wb") as outgoing:
            shutil.copyfileobj(incoming, outgoing, length=8 * 1024 * 1024)
            flush_file(outgoing)
        if file_hash(temp) != checksum:
            raise OSError("保存ファイルのSHA-256が一致しません。")
        os.replace(temp, destination)
    finally:
        temp.unlink(missing_ok=True)
    return checksum


def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def rng_state():
    np_state = np.random.get_state()
    return dict(python=random.getstate(), torch=torch.get_rng_state().clone(),
                cuda=[x.clone().cpu() for x in torch.cuda.get_rng_state_all()]
                if torch.cuda.is_available() else [],
                numpy=[np_state[0], np_state[1].tolist(), np_state[2], np_state[3], np_state[4]])


def restore_rng(state):
    random.setstate(state["python"])
    torch.set_rng_state(state["torch"].cpu())
    np_state = state["numpy"]
    np.random.set_state((np_state[0], np.asarray(np_state[1], dtype=np.uint32),
                         np_state[2], np_state[3], np_state[4]))
    if state["cuda"] and torch.cuda.is_available():
        if len(state["cuda"]) != torch.cuda.device_count():
            raise ValueError("保存時とGPU台数が違います。1GPUのランタイムを選んでください。")
        torch.cuda.set_rng_state_all(state["cuda"])


def cpu_tree(value):
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().clone()
    if isinstance(value, dict):
        return {key: cpu_tree(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return type(value)(cpu_tree(item) for item in value)
    return value


class LoRALinear(nn.Module):
    """Frozen Linear + FP32 trainable factors, without mutating shared cores."""
    def __init__(self, base, rank, alpha):
        super().__init__()
        self.base = base.requires_grad_(False)
        self.in_features, self.out_features = base.in_features, base.out_features
        self.rank, self.alpha = int(rank), float(alpha)
        self.scaling = self.alpha / self.rank
        self.enabled = True
        self.lora_A = nn.Parameter(torch.empty(self.rank, self.in_features,
                                               device=base.weight.device, dtype=torch.float32))
        self.lora_B = nn.Parameter(torch.zeros(self.out_features, self.rank,
                                               device=base.weight.device, dtype=torch.float32))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))

    @property
    def weight(self):
        return self.base.weight

    @property
    def bias(self):
        return self.base.bias

    def forward(self, x):
        result = self.base(x)
        if self.enabled:
            update = F.linear(F.linear(x.to(self.lora_A.dtype), self.lora_A), self.lora_B)
            result = result + (update * self.scaling).to(result.dtype)
        return result


def lora_modules(model):
    return {name: module for name, module in model.named_modules() if isinstance(module, LoRALinear)}


def inject_lora(model, rank=16, alpha=16):
    if rank <= 0 or alpha <= 0:
        raise ValueError("rankとalphaは正の値にしてください。")
    if lora_modules(model):
        raise ValueError("このモデルには既にLoRAが入っています。読み込みセルから再実行してください。")
    model.requires_grad_(False)
    targets = []
    for name, module in list(model.named_modules()):
        if not isinstance(module, nn.Linear):
            continue
        if not name.startswith(("blocks.", "pixel_blocks.", "y_embedder.", "s_embedder.")):
            continue
        if any(part.startswith("adaln") for part in name.split(".")):
            continue
        if min(module.in_features, module.out_features) < 16:
            continue
        parent_name, _, child_name = name.rpartition(".")
        parent = model.get_submodule(parent_name) if parent_name else model
        setattr(parent, child_name, LoRALinear(module, rank, alpha))
        targets.append(name)
    if not targets:
        raise ValueError("LoRAを適用できるLinearが見つかりません。Irisのrevisionを確認してください。")
    return targets


def adapter_state(model):
    return {name: tensor.detach().float().cpu().contiguous().clone()
            for name, tensor in model.named_parameters() if name.endswith((".lora_A", ".lora_B"))}


def load_adapter_state(model, state):
    live = {name: parameter for name, parameter in model.named_parameters()
            if name.endswith((".lora_A", ".lora_B"))}
    if set(live) != set(state) or any(live[k].shape != state[k].shape for k in live):
        raise ValueError("保存LoRAとモデルのLinear名・shapeが一致しません。")
    with torch.no_grad():
        for name, parameter in live.items():
            parameter.copy_(state[name].to(device=parameter.device, dtype=parameter.dtype))


def merged_state_dict(model):
    modules = lora_modules(model)
    replacements = {}
    updates = {}
    for name, module in modules.items():
        replacements[name + ".base.weight"] = name + ".weight"
        if module.base.bias is not None:
            replacements[name + ".base.bias"] = name + ".bias"
        updates[name + ".weight"] = (module.lora_B.detach().float().cpu()
                                      @ module.lora_A.detach().float().cpu()) * module.scaling
    output = {}
    for name, tensor in model.state_dict().items():
        if name.endswith((".lora_A", ".lora_B")):
            continue
        target = replacements.get(name, name)
        base = tensor.detach().cpu()
        if target in updates:
            base = (base.float() + updates[target]).to(base.dtype)
        output[target] = base.contiguous().clone()
    return output


def dataset_config(settings, *, require_source=False):
    """Bind a user-selected dataset, commit, Parquet pattern and column mapping."""
    repo = str(settings.get("dataset_repo", "")).strip()
    revision = str(settings.get("dataset_revision", "")).strip()
    columns = dict(DEFAULT_DATA_COLUMNS)
    overrides = settings.get("dataset_columns", {})
    if not isinstance(overrides, dict) or set(overrides) - set(columns):
        raise ValueError("dataset_columns contains unsupported column keys.")
    columns.update(overrides)
    for key, value in columns.items():
        if key == "aesthetic_tier" and value in (None, ""):
            columns[key] = None
        elif not isinstance(value, str) or not value.strip():
            raise ValueError(f"Set a nonempty dataset column for {key}.")
        else:
            columns[key] = value.strip()
    pattern = settings.get("parquet_glob", "*.parquet")
    if not isinstance(pattern, str) or not pattern.strip():
        raise ValueError("parquet_glob must be a nonempty file pattern.")
    if require_source:
        if not re.fullmatch(r"[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+", repo):
            raise ValueError("Set DATASET_REPO to your Hugging Face dataset repository.")
        if not re.fullmatch(r"[0-9a-fA-F]{40}", revision):
            raise ValueError("Set DATASET_REVISION to the dataset's full 40-character commit SHA.")
        if columns["aesthetic_tier"] is None and settings.get("min_aesthetic_tier", 0) > 0:
            raise ValueError("Set QUALITY_COLUMN, or use MIN_AESTHETIC_TIER=0 without a quality column.")
    return dict(repo=repo, revision=revision, columns=columns, parquet_glob=pattern.strip())


def data_recipe(settings):
    recipe = {key: settings[key] for key in ("seed", "resolution", "aspect_buckets",
              "sample_count", "min_aesthetic_tier", "sfw_only")}
    recipe.update(schema=SCHEMA_VERSION, **dataset_config(settings),
                  selection="seeded-shards-shuffled-batches-v1", image_transform="lanczos-center-fit-v1")
    if "exclusion" in settings:
        recipe["exclusion"] = copy.deepcopy(settings["exclusion"])
        recipe["selection"] = "seeded-shards-unseen-images-v1"
    return recipe


def flow_shift_for_resolution(resolution):
    """Use the official 512/1024 training-stage noise schedules."""
    if resolution not in (512, 1024):
        raise ValueError("学習解像度は512または1024を選んでください。")
    return 3.0 if resolution == 512 else 4.0


class PreparedDataset:
    def __init__(self, root, records, fingerprint, drive_dir=None):
        self.root, self.records, self.fingerprint = Path(root), records, fingerprint
        self.drive_dir = Path(drive_dir) if drive_dir else None

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        row = self.records[index]
        with Image.open(self.root / row["path"]) as source:
            image = ImageOps.exif_transpose(source).convert("RGB")
            image = ImageOps.fit(image, (row["width"], row["height"]),
                                 method=Image.Resampling.LANCZOS, centering=(0.5, 0.5))
            pixels = np.asarray(image, dtype=np.float32).copy()
        return torch.from_numpy(pixels).permute(2, 0, 1).contiguous().div_(127.5).sub_(1), row["prompt"]


def dataset_rows(settings, scratch):
    """Download only the seeded shards consumed, decode bounded Arrow batches."""
    config = dataset_config(settings, require_source=True)
    from huggingface_hub import HfApi, hf_hub_download
    import pyarrow.parquet as pq
    files = [name for name in HfApi().list_repo_files(config["repo"], repo_type="dataset", revision=config["revision"])
             if name.endswith(".parquet") and fnmatch.fnmatchcase(name, config["parquet_glob"])]
    files.sort()
    rng = random.Random(settings["seed"])
    rng.shuffle(files)
    if not files:
        raise RuntimeError("データセットにParquetシャードがありません。")
    cache = Path(scratch) / "shard-downloads"
    for name in files:
        print(f"データを読み込み中: {name}", flush=True)
        path = Path(hf_hub_download(config["repo"], name, repo_type="dataset", revision=config["revision"],
                                    local_dir=str(cache)))
        parquet = pq.ParquetFile(path, memory_map=True)
        try:
            selected_columns = list(dict.fromkeys(name for name in config["columns"].values() if name))
            missing = set(selected_columns) - set(parquet.schema_arrow.names)
            if missing:
                raise ValueError(f"Missing Parquet columns in {name}: {sorted(missing)}")
            for batch in parquet.iter_batches(batch_size=64, columns=selected_columns):
                rows = batch.to_pylist()
                rng.shuffle(rows)
                for row in rows:
                    mapped = {key: row[column] if column else 0 for key, column in config["columns"].items()}
                    raw = mapped["image"]
                    if isinstance(raw, dict):
                        raw = raw.get("bytes")
                    if not isinstance(raw, (bytes, bytearray, memoryview)):
                        raise ValueError("The image column must contain embedded bytes, or an HF Image bytes struct.")
                    mapped["image"] = bytes(raw)
                    yield mapped
        finally:
            parquet.close()
            # Only this task-owned download is removed; selected images are retained.
            path.unlink(missing_ok=True)


def _is_sfw(prompt):
    tags = {tag.strip().lower() for tag in prompt.split(",")}
    allowed = {"safe", "general", "rating:general", "rating:safe", "rating:g"}
    excluded = {"sensitive", "questionable", "explicit", "rating:s", "rating:q", "rating:e"}
    return bool(tags & allowed) and not bool(tags & excluded)


def _restore_dataset(drive_dir, local_dir, manifest, recipe):
    if manifest.get("recipe") != recipe or len(manifest.get("records", [])) != recipe["sample_count"]:
        raise ValueError("保存データの設定・画像数が一致しません。")
    expected = stable_hash({"recipe": recipe, "records": manifest["records"]})
    if manifest.get("fingerprint") != expected:
        raise RuntimeError("保存データのmanifestが破損しています。")
    archive_name = manifest["archive"]
    if not re.fullmatch(r"dataset-[a-f0-9]{12}\.tar", archive_name):
        raise RuntimeError("保存データのarchive名が不正です。")
    records = manifest["records"]
    if any(not IMAGE_NAME.fullmatch(row["path"]) for row in records):
        raise RuntimeError("保存データの画像パスが不正です。")
    ready_path = local_dir / "ready.json"
    ready = json.loads(ready_path.read_text()) if ready_path.is_file() else {}
    if ready.get("fingerprint") != expected or not all((local_dir / row["path"]).is_file() for row in records):
        local_dir.mkdir(parents=True, exist_ok=True)
        archive = local_dir / "restore.tar"
        try:
            verified_copy(drive_dir / archive_name, archive, manifest["archive_sha256"])
        except (OSError, FileNotFoundError) as exc:
            raise RuntimeError("保存データのアーカイブが破損または未保存です。") from exc
        with tarfile.open(archive, "r") as pack:
            members = pack.getmembers()
            if (len(members) != len(records) or
                {item.name for item in members} != {row["path"] for row in records} or
                any(not item.isfile() or not IMAGE_NAME.fullmatch(item.name) for item in members)):
                raise RuntimeError("保存データのアーカイブ内容がmanifestと一致しません。")
            for member in members:
                destination = local_dir / member.name
                destination.parent.mkdir(parents=True, exist_ok=True)
                with pack.extractfile(member) as incoming, destination.open("wb") as outgoing:
                    shutil.copyfileobj(incoming, outgoing)
        archive.unlink()
        atomic_json(ready_path, {"fingerprint": expected})
    return PreparedDataset(local_dir, records, expected, drive_dir)


def prepare_dataset(drive_root, scratch_root, settings, source_factory=None, exclusion=None):
    recipe = data_recipe(settings)
    excluded_ids, excluded_hashes = set(), set()
    if "exclusion" in recipe:
        if exclusion is None or stable_hash(exclusion) != recipe["exclusion"]["sha256"]:
            raise ValueError("The exact prior-image exclusion ledger is required.")
        excluded_ids = set(exclusion["image_ids"])
        excluded_hashes = set(exclusion["image_sha256"])
    elif exclusion is not None:
        raise ValueError("Image exclusions must be included in the dataset recipe.")
    if recipe["sample_count"] <= 0:
        raise ValueError("sample_countは1以上にしてください。")
    flow_shift_for_resolution(recipe["resolution"])
    key = stable_hash(recipe)[:16]
    drive_dir = Path(drive_root) / "datasets" / key
    local_dir = Path(scratch_root) / "datasets" / key
    manifest_path = drive_dir / "manifest.json"
    if manifest_path.is_file():
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        result = _restore_dataset(drive_dir, local_dir, manifest, recipe)
        if any(row["image_id"] in excluded_ids or row["image_sha256"] in excluded_hashes
               for row in result.records):
            raise RuntimeError("Saved data overlaps the prior training pools.")
        print(f"保存データを復元: {len(result):,}枚", flush=True)
        return result
    from iris3b.data.buckets import SHARED_21_512, SHARED_21_1024, closest_ratio
    buckets = SHARED_21_512 if recipe["resolution"] == 512 else SHARED_21_1024
    local_dir.mkdir(parents=True, exist_ok=True)
    (local_dir / "images").mkdir(exist_ok=True)
    records, seen, seen_hashes, skipped = [], set(), set(), 0
    source = source_factory() if source_factory else dataset_rows(settings, scratch_root)
    try:
        for row in source:
            quality = int(row["aesthetic_tier"])
            prompt = str(row.get("prompt") or "").strip()
            identity = "" if row.get("image_id") is None else str(row["image_id"])
            if (quality < recipe["min_aesthetic_tier"] or not prompt or not identity or identity in seen or
                identity in excluded_ids or
                (recipe["sfw_only"] and not _is_sfw(prompt))):
                skipped += 1
                continue
            raw = row["image"]
            image_hash = hashlib.sha256(raw).hexdigest()
            if image_hash in excluded_hashes or (exclusion is not None and image_hash in seen_hashes):
                skipped += 1
                continue
            try:
                with Image.open(io.BytesIO(raw)) as original:
                    oriented = ImageOps.exif_transpose(original)
                    oriented.load()
                    width, height = oriented.size
                    if width <= 0 or height <= 0:
                        raise ValueError("empty image")
            except (OSError, ValueError, Image.DecompressionBombError):
                skipped += 1
                continue
            if recipe["aspect_buckets"]:
                _, (target_h, target_w) = closest_ratio(height, width, buckets)
            else:
                target_h = target_w = recipe["resolution"]
            image_name = f"images/{len(records):06d}.bin"
            (local_dir / image_name).write_bytes(raw)
            records.append(dict(path=image_name, prompt=prompt, image_id=identity,
                aesthetic_tier=quality, height=target_h, width=target_w,
                original_height=height, original_width=width, image_sha256=image_hash))
            seen.add(identity)
            seen_hashes.add(image_hash)
            if len(records) % 500 == 0:
                print(f"高品質画像: {len(records):,}/{recipe['sample_count']:,}", flush=True)
            if len(records) == recipe["sample_count"]:
                break
    finally:
        if hasattr(source, "close"):
            source.close()
    if len(records) != recipe["sample_count"]:
        raise RuntimeError(f"高品質画像が不足しています: {len(records)}/{recipe['sample_count']}枚。")
    fingerprint = stable_hash({"recipe": recipe, "records": records})
    archive_name = "dataset-" + uuid.uuid4().hex[:12] + ".tar"
    archive = local_dir / archive_name
    with tarfile.open(archive, "w") as pack:
        for row in records:
            pack.add(local_dir / row["path"], arcname=row["path"], recursive=False)
    checksum = verified_copy(archive, drive_dir / archive_name)
    manifest = dict(recipe=recipe, records=records, fingerprint=fingerprint,
                    archive=archive_name, archive_sha256=checksum)
    atomic_json(manifest_path, manifest)
    atomic_json(local_dir / "ready.json", {"fingerprint": fingerprint})
    archive.unlink()
    print(f"抽出完了: {len(records):,}枚、除外{skipped:,}件。データをDriveへ保存しました。", flush=True)
    return PreparedDataset(local_dir, records, fingerprint, drive_dir)


def check_a100():
    if not torch.cuda.is_available():
        raise RuntimeError("ColabのランタイムをGPU / A100に変更してください。")
    name = torch.cuda.get_device_name(0)
    memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
    if "A100" not in name or memory < 35 or not torch.cuda.is_bf16_supported():
        raise RuntimeError(f"A100・35GiB以上・BF16対応が必要です。現在: {name}, {memory:.1f}GiB")
    if torch.cuda.device_count() != 1:
        raise RuntimeError("1台のA100を持つColabランタイムを選んでください。")
    print(f"GPU: {name} / {memory:.1f}GiB / PyTorch {torch.__version__}")


def load_models(settings, cache_root, model_dir=None, text_dir=None, device="cuda"):
    shift = flow_shift_for_resolution(settings["resolution"])
    from huggingface_hub import snapshot_download
    from omegaconf import OmegaConf
    from safetensors.torch import load_file
    from iris3b.config import inference_config
    from iris3b.models import IrisDiT
    from iris3b.text.qwen3_vl import Qwen3VLTextEncoder
    seed_all(settings["seed"])
    cache_root = Path(cache_root)
    if model_dir is None:
        model_dir = snapshot_download(BASE_REPO, revision=BASE_REVISION,
            allow_patterns=["config.yaml", "model.safetensors"], local_dir=str(cache_root / "iris-base"))
    if text_dir is None:
        text_dir = snapshot_download(TEXT_REPO, revision=TEXT_REVISION,
            allow_patterns=["*.json", "*.safetensors", "merges.txt", "vocab.json", "tokenizer*"],
            local_dir=str(cache_root / "qwen-text"))
    cfg = inference_config(OmegaConf.to_container(OmegaConf.load(Path(model_dir) / "config.yaml")))
    cfg.flow.shift, cfg.flow.shift_law = shift, "none"
    cfg.text_encoder.pretrained = str(text_dir)
    cfg.text_encoder.null_embed_dir = str(cache_root / "null-text")
    cfg.text_encoder.compile = False
    weights = load_file(str(Path(model_dir) / "model.safetensors"), device="cpu")
    with torch.device("meta"):
        model = IrisDiT(cfg.model)
    model.load_state_dict(weights, strict=True, assign=True)
    del weights
    model.to(device=device, dtype=torch.bfloat16).requires_grad_(False)
    model.activation_checkpointing = "full"
    targets = inject_lora(model, settings["rank"], settings["alpha"])
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    encoder = Qwen3VLTextEncoder(cfg.text_encoder, device=device)
    # Weights/tokenizer are already loaded from the pinned local snapshot.
    # Upstream derives the null-cache filename from pretrained; use the model
    # identity rather than a filesystem path (also valid on Windows).
    cfg.text_encoder.pretrained = TEXT_REPO
    count = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
    print(f"LoRA対象: {len(targets)} Linear / 学習パラメータ: {count:,} / Qwenは凍結")
    return model, encoder, cfg, targets


class LoRATrainer:
    checkpoint_tensor_key = "adapter"
    gradient_clip = 1.0
    micro_batch_size = 1

    def __init__(self, model, encoder, cfg, targets, dataset, settings, run_dir, scratch, device="cuda"):
        from iris3b.flow import RectifiedFlow
        self.model, self.encoder, self.cfg = model, encoder, cfg
        self.dataset, self.settings = dataset, dict(settings)
        self.run_dir, self.scratch, self.device = Path(run_dir), Path(scratch), torch.device(device)
        self.run_dir.mkdir(parents=True, exist_ok=True)
        self.scratch.mkdir(parents=True, exist_ok=True)
        if len(dataset) != settings["sample_count"] or len(dataset) == 0:
            raise ValueError("データの画像数と学習設定が一致しません。データ準備セルから再実行してください。")
        if self.settings["grad_accum"] < 1 or self.settings["learning_rate"] <= 0:
            raise ValueError("grad_accumとlearning_rateは正の値にしてください。")
        parameters = [p for p in model.parameters() if p.requires_grad]
        self._validate_trainable_model(parameters)
        self.optimizer = self._create_optimizer(parameters)
        warmup = max(1, int(settings["warmup_steps"]))
        self.scheduler = torch.optim.lr_scheduler.LambdaLR(self.optimizer,
            lambda step: min((step + 1) / warmup, 1.0))
        self.flow = RectifiedFlow(cfg.flow, tokens=(settings["resolution"] // cfg.model.patch_size)**2)
        self.recipe = self._build_recipe(targets)
        self.signature = stable_hash(self.recipe)
        self.step, self.cursor, self.last_loss = 0, 0, None
        self.last_saved_step, self.last_saved_record = None, None
        self.invalid_optimizer_state = False
        self._epoch, self._order = None, None
        # Build the fixed null once; do not keep per-image 12-layer embeddings on disk.
        self.null = self._build_null()
        self.committed_rng = rng_state()

    def _validate_trainable_model(self, parameters):
        if not parameters or any(p.requires_grad and not name.endswith((".lora_A", ".lora_B"))
                                 for name, p in self.model.named_parameters()):
            raise ValueError("学習対象はLoRAパラメータだけにしてください。")

    def _create_optimizer(self, parameters):
        return torch.optim.AdamW(parameters, lr=self.settings["learning_rate"],
            betas=(0.9, 0.95), weight_decay=0, fused=self.device.type == "cuda")

    def _build_recipe(self, targets):
        keys = ("seed", "learning_rate", "warmup_steps", "grad_accum", "text_dropout", "rank", "alpha")
        return dict(schema=SCHEMA_VERSION, source_revision=SOURCE_REVISION,
            base_repo=BASE_REPO, base_revision=BASE_REVISION, text_repo=TEXT_REPO,
            text_revision=TEXT_REVISION, dataset_fingerprint=self.dataset.fingerprint,
            data=data_recipe(self.settings), training={key: self.settings[key] for key in keys}, targets=list(targets),
            model=asdict(self.cfg.model), flow=asdict(self.cfg.flow), base_dtype=str(next(self.model.parameters()).dtype))

    def _build_null(self):
        return self.encoder.null()

    def _checkpoint_weights(self):
        return adapter_state(self.model)

    def _restore_weights(self, state):
        load_adapter_state(self.model, state)

    def _prepare_step(self):
        pass

    def _clear_step(self):
        pass

    def _autocast(self):
        return torch.autocast(self.device.type, dtype=torch.bfloat16, enabled=self.device.type == "cuda")

    def index_at(self, cursor):
        epoch, offset = divmod(cursor, len(self.dataset))
        if epoch != self._epoch:
            self._order = list(range(len(self.dataset)))
            random.Random(self.settings["seed"] + epoch).shuffle(self._order)
            self._epoch = epoch
        return self._order[offset]

    def micro_loss(self, cursor):
        image, caption = self.dataset[self.index_at(cursor)]
        image = image.unsqueeze(0).to(self.device, dtype=torch.float32)
        # The encoder's hidden states must be ordinary detached tensors: an
        # inference-mode tensor cannot be saved for the trainable text adapter.
        with torch.no_grad():
            drop = torch.rand((), device=self.device).item() < self.settings["text_dropout"]
            encoding = self.null if drop else self.encoder.encode([caption])
            embedding = encoding.embeddings.detach().to(self.device)
            mask = encoding.mask.detach().to(self.device)
        with self._autocast():
            return self.flow.training_loss(self.model, image, embedding,
                                            model_kwargs={"y_mask": mask}).loss

    def sanity_check(self):
        state = rng_state()
        self.model.train()
        self.optimizer.zero_grad(set_to_none=True)
        if self.device.type == "cuda":
            torch.cuda.reset_peak_memory_stats(self.device)
        try:
            loss = self.micro_loss(self.cursor)
            if not torch.isfinite(loss):
                raise RuntimeError("最小確認のlossが非有限です。")
            loss.backward()
            gradients = [p.grad for p in self.model.parameters() if p.requires_grad and p.grad is not None]
            if not gradients or not all(torch.isfinite(grad).all() for grad in gradients):
                raise RuntimeError("学習パラメータの勾配が無効です。")
            if not any(torch.count_nonzero(grad) for grad in gradients):
                raise RuntimeError("学習パラメータの勾配が全て0です。")
            if any(p.grad is not None for p in self.model.parameters() if not p.requires_grad):
                raise RuntimeError("凍結ベースに勾配が発生しています。")
            memory = torch.cuda.max_memory_allocated(self.device) / 1024**3 if self.device.type == "cuda" else 0
            print(f"forward/backward OK / loss {loss.item():.5f} / peak allocated {memory:.2f}GiB")
        finally:
            self.optimizer.zero_grad(set_to_none=True)
            restore_rng(state)

    def save(self):
        if self.invalid_optimizer_state:
            raise RuntimeError("optimizer更新が中断されたモデルは保存できません。読み込みセルから再実行してください。")
        if self.last_saved_step == self.step:
            return self.last_saved_record
        directory = self.run_dir / "checkpoints"
        directory.mkdir(parents=True, exist_ok=True)
        filename = f"step-{self.step:08d}-{uuid.uuid4().hex[:8]}.pt"
        local_file = self.scratch / filename
        payload = dict(schema=SCHEMA_VERSION, signature=self.signature, recipe=self.recipe,
            optimizer=cpu_tree(self.optimizer.state_dict()),
            scheduler=self.scheduler.state_dict(), step=self.step, cursor=self.cursor,
            rng=self.committed_rng, last_loss=self.last_loss,
            runtime=dict(torch=str(torch.__version__), device_type=self.device.type))
        payload[self.checkpoint_tensor_key] = self._checkpoint_weights()
        torch.save(payload, local_file)
        del payload
        try:
            checksum = verified_copy(local_file, directory / filename)
        finally:
            local_file.unlink(missing_ok=True)
        index_path = directory / "index.json"
        index = json.loads(index_path.read_text()) if index_path.is_file() else {"schema": SCHEMA_VERSION, "checkpoints": []}
        item = dict(file=filename, sha256=checksum, step=self.step, cursor=self.cursor)
        existing = index["checkpoints"]
        existing.append(item)
        keep = max(2, int(self.settings["keep_checkpoints"]))
        retained, removed = existing[-keep:], existing[:-keep]
        atomic_json(index_path, {"schema": SCHEMA_VERSION, "checkpoints": retained})
        self.last_saved_step, self.last_saved_record = self.step, item
        for old in removed:
            if CHECKPOINT_NAME.fullmatch(old["file"]):
                (directory / old["file"]).unlink(missing_ok=True)
        print(f"Driveへ保存: step {self.step:,} / 使用画像 {self.cursor:,}", flush=True)
        return item

    def resume(self):
        directory = self.run_dir / "checkpoints"
        index_path = directory / "index.json"
        if not index_path.is_file():
            # An uncommitted copy is not a successfully saved checkpoint.
            if any(directory.glob("step-*.pt")):
                raise RuntimeError("チェックポイントのindexがありません。保存結果を確認してください。")
            print("新規学習を開始します。")
            return 0
        try:
            index = json.loads(index_path.read_text())
            entries = index["checkpoints"]
            if index.get("schema") != SCHEMA_VERSION or not entries:
                raise ValueError("invalid index")
        except (ValueError, KeyError, TypeError) as exc:
            raise RuntimeError("チェックポイントのindexが破損しています。") from exc
        for item in reversed(entries):
            if not CHECKPOINT_NAME.fullmatch(item["file"]):
                continue
            source = directory / item["file"]
            local = self.scratch / ("resume-" + item["file"])
            try:
                verified_copy(source, local, item["sha256"])
                payload = torch.load(local, map_location="cpu", weights_only=True, mmap=os.name != "nt")
            except (OSError, RuntimeError, EOFError, ValueError, pickle.UnpicklingError) as exc:
                print(f"破損した保存を除外: step {item['step']} ({type(exc).__name__})")
                continue
            finally:
                local.unlink(missing_ok=True)
            if payload.get("signature") != self.signature or payload.get("recipe") != self.recipe:
                raise ValueError("再開設定またはデータが保存時と違います。設定を戻すかRUN_NAMEを変更してください。")
            if (payload.get("schema") != SCHEMA_VERSION or payload["step"] != item["step"] or
                payload["cursor"] != item["cursor"] or
                payload["cursor"] != payload["step"] * self.settings["grad_accum"] * self.micro_batch_size):
                raise RuntimeError("チェックポイントのstep・データ位置が一致しません。")
            self._restore_weights(payload[self.checkpoint_tensor_key])
            self.optimizer.load_state_dict(payload["optimizer"])
            self.scheduler.load_state_dict(payload["scheduler"])
            self.step, self.cursor = payload["step"], payload["cursor"]
            self.last_loss, self.committed_rng = payload["last_loss"], payload["rng"]
            self.last_saved_step, self.last_saved_record = self.step, item
            self.invalid_optimizer_state = False
            restore_rng(self.committed_rng)
            self._epoch = None
            print(f"再開: step {self.step:,} / 使用画像 {self.cursor:,}", flush=True)
            return self.step
        raise RuntimeError("正常なチェックポイントが見つかりません。保存ファイルを確認してください。")

    def update_runtime_settings(self, settings):
        frozen_keys = self._immutable_setting_keys()
        if (any(self.settings[key] != settings[key] for key in frozen_keys)
                or data_recipe(self.settings) != data_recipe(settings)):
            raise ValueError("学習設定が読み込み時から変わっています。RUN_NAMEを変更し、データとモデルを読み直してください。")
        for key in ("max_steps", "session_minutes", "save_every", "save_minutes"):
            if settings[key] <= 0:
                raise ValueError("ステップ数・時間・保存間隔は正の値にしてください。")
            self.settings[key] = settings[key]

    def _immutable_setting_keys(self):
        return ("seed", "learning_rate", "warmup_steps", "grad_accum", "text_dropout", "rank", "alpha",
                "sample_count", "resolution", "aspect_buckets", "min_aesthetic_tier", "sfw_only")

    def train(self, max_steps=None):
        if self.invalid_optimizer_state:
            raise RuntimeError("optimizer更新が中断されています。読み込みセルから再実行してください。")
        max_steps = int(max_steps if max_steps is not None else self.settings["max_steps"])
        if self.step >= max_steps:
            print(f"目標の{max_steps:,}ステップまで完了しています。")
            return
        started = last_save = time.monotonic()
        accumulation = self.settings["grad_accum"]
        self.model.train()
        self.optimizer.zero_grad(set_to_none=True)
        phase = "accumulation"
        try:
            while self.step < max_steps:
                loss_sum = 0.0
                phase = "accumulation"
                self._prepare_step()
                for micro in range(accumulation):
                    loss = self.micro_loss(self.cursor + micro * self.micro_batch_size)
                    if not torch.isfinite(loss):
                        raise RuntimeError(f"step {self.step}のlossが非有限です。前回保存から再開してください。")
                    (loss / accumulation).backward()
                    loss_sum += float(loss.detach())
                    del loss
                trainable = [p for p in self.model.parameters() if p.requires_grad]
                norm = torch.nn.utils.clip_grad_norm_(trainable, self.gradient_clip, error_if_nonfinite=True)
                phase = "optimizer"
                self.optimizer.step()
                self.scheduler.step()
                self.optimizer.zero_grad(set_to_none=True)
                self._clear_step()
                self.step += 1
                self.cursor += accumulation * self.micro_batch_size
                self.last_loss = loss_sum / accumulation
                self.committed_rng = rng_state()
                phase = "committed"
                now = time.monotonic()
                if self.step == 1 or self.step % 10 == 0:
                    minutes = (now - started) / 60
                    print(f"step {self.step:,}/{max_steps:,} / loss {self.last_loss:.5f} / "
                          f"grad {float(norm):.3f} / {minutes:.1f}分", flush=True)
                if (self.step % self.settings["save_every"] == 0 or
                    now - last_save >= 60 * self.settings["save_minutes"]):
                    self.save()
                    last_save = time.monotonic()
                if now - started >= 60 * self.settings["session_minutes"]:
                    print("この実行の時間上限に達しました。再実行で続けられます。")
                    break
            self.save()
        except KeyboardInterrupt:
            self.optimizer.zero_grad(set_to_none=True)
            if phase != "optimizer":
                restore_rng(self.committed_rng)
                self.save()
                print("停止しました。完了したoptimizerステップまで保存済みです。")
            else:
                self.invalid_optimizer_state = True
                print("optimizer更新中に停止しました。前回の正常な保存を保持します。読み込みセルから再実行してください。")
        except Exception:
            self.optimizer.zero_grad(set_to_none=True)
            if phase != "optimizer":
                restore_rng(self.committed_rng)
                try:
                    self.save()
                except Exception as save_error:
                    print(f"今回の緊急保存に失敗しました: {type(save_error).__name__}。前回の保存から再開してください。")
            else:
                self.invalid_optimizer_state = True
            raise
        finally:
            self._clear_step()

    @torch.no_grad()
    def sample(self, prompts, steps=50, seed=42, compare_base=True):
        if self.invalid_optimizer_state:
            raise RuntimeError("optimizer更新が中断されています。読み込みセルから再実行してください。")
        from iris3b.sampling import generate
        state = rng_state()
        was_training = self.model.training
        modules = lora_modules(self.model)
        self.model.eval()
        directory = self.run_dir / "samples"
        directory.mkdir(parents=True, exist_ok=True)
        paths = []
        try:
            modes = [False, True] if compare_base else [True]
            for enabled in modes:
                for module in modules.values():
                    module.enabled = enabled
                label = f"step-{self.step:08d}" if enabled else "base"
                for index, prompt in enumerate(prompts):
                    gen = torch.Generator(device=self.device).manual_seed(seed + index)
                    with self._autocast():
                        output = generate(self.model, self.encoder, [prompt],
                            height=self.settings["resolution"], width=self.settings["resolution"],
                            steps=steps, cfg_scale=3.0, shift=self.flow.shift, generator=gen,
                            device=self.device, num_train_timesteps=self.cfg.flow.num_train_timesteps,
                            prediction=self.cfg.flow.prediction)[0]
                    pixels = output.detach().float().cpu().add(1).mul(127.5).clamp(0, 255)
                    image = Image.fromarray(pixels.permute(1, 2, 0).round().to(torch.uint8).numpy())
                    local = self.scratch / f"{label}-{index:02d}.png"
                    image.save(local)
                    target = directory / local.name
                    verified_copy(local, target)
                    paths.append(target)
            atomic_json(directory / f"step-{self.step:08d}-prompts.json",
                        dict(prompts=prompts, seed=seed, steps=steps, cfg_scale=3.0, shift=self.flow.shift))
        finally:
            for module in modules.values():
                module.enabled = True
            self.model.train(was_training)
            restore_rng(state)
        return paths


def export_adapter(model, output_dir, recipe, step):
    from safetensors.torch import save_file
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    temp = output_dir / ("adapter-" + uuid.uuid4().hex + ".partial")
    try:
        save_file(adapter_state(model), str(temp), metadata={"format": "iris3b-linear-lora-v1"})
        os.replace(temp, output_dir / "adapter.safetensors")
    finally:
        temp.unlink(missing_ok=True)
    config = copy.deepcopy(recipe)
    config.update(format="iris3b-linear-lora-v1", optimizer_step=step)
    atomic_json(output_dir / "adapter_config.json", config)
    print(f"LoRAを書き出しました: {output_dir}")
    return output_dir


def export_merged(model, cfg, output_dir):
    from omegaconf import OmegaConf
    from safetensors.torch import save_file
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    weights = merged_state_dict(model)
    temp = output_dir / ("model-" + uuid.uuid4().hex + ".partial")
    try:
        save_file(weights, str(temp))
        os.replace(temp, output_dir / "model.safetensors")
    finally:
        temp.unlink(missing_ok=True)
    text = asdict(cfg.text_encoder)
    text["pretrained"], text["null_embed_dir"] = TEXT_REPO, "output/pretrained_models"
    OmegaConf.save(OmegaConf.create(dict(model=asdict(cfg.model), text_encoder=text,
                                        flow=asdict(cfg.flow))), output_dir / "config.yaml")
    print(f"公式Iris形式へマージしました: {output_dir}")
    return output_dir


In [ ]:
%%writefile /content/iris_full_ft_colab.py
"""Full Iris fine-tuning for one A100, sharing data/checkpoint primitives."""
from __future__ import annotations

import gc
import importlib.metadata
import math
import random
import shutil
from dataclasses import asdict
from pathlib import Path

import torch

import iris_lora_colab as core

BITSANDBYTES_VERSION = "0.49.2"


class BucketBatchOrder:
    """Deterministic full batches; cover every image, then pad only bucket tails."""

    def __init__(self, records, batch_size, seed):
        if not isinstance(batch_size, int) or isinstance(batch_size, bool) or batch_size < 1:
            raise ValueError("micro_batch_sizeは1以上の整数にしてください。")
        groups = {}
        for index, record in enumerate(records):
            groups.setdefault((record["height"], record["width"]), []).append(index)
        self.groups = [groups[key] for key in sorted(groups)]
        self.batch_size, self.seed = batch_size, seed
        self.epoch_size = sum(math.ceil(len(group) / batch_size) * batch_size for group in self.groups)
        self.padding = self.epoch_size - len(records)

    def order(self, epoch):
        rng = random.Random(self.seed + epoch)
        batches = []
        for group in self.groups:
            indices = list(group)
            rng.shuffle(indices)
            missing = (-len(indices)) % self.batch_size
            indices.extend(indices[i % len(group)] for i in range(missing))
            batches.extend(indices[offset:offset + self.batch_size]
                           for offset in range(0, len(indices), self.batch_size))
        rng.shuffle(batches)
        return [index for batch in batches for index in batch]


def load_full_models(settings, cache_root, model_dir=None, text_dir=None, device="cuda"):
    from huggingface_hub import snapshot_download
    from omegaconf import OmegaConf
    from safetensors.torch import load_file
    from iris3b.config import inference_config
    from iris3b.models import IrisDiT
    from iris3b.text.qwen3_vl import Qwen3VLTextEncoder

    shift = core.flow_shift_for_resolution(settings["resolution"])
    core.seed_all(settings["seed"])
    cache_root = Path(cache_root)
    if model_dir is None:
        model_dir = snapshot_download(core.BASE_REPO, revision=core.BASE_REVISION,
            allow_patterns=["config.yaml", "model.safetensors"], local_dir=str(cache_root / "iris-base"))
    if text_dir is None:
        text_dir = snapshot_download(core.TEXT_REPO, revision=core.TEXT_REVISION,
            allow_patterns=["*.json", "*.safetensors", "merges.txt", "vocab.json", "tokenizer*"],
            local_dir=str(cache_root / "qwen-text"))
    cfg = inference_config(OmegaConf.to_container(OmegaConf.load(Path(model_dir) / "config.yaml")))
    cfg.flow.shift, cfg.flow.shift_law = shift, "none"
    cfg.text_encoder.pretrained = str(text_dir)
    cfg.text_encoder.null_embed_dir = str(cache_root / "null-text")
    cfg.text_encoder.compile = False
    weights = load_file(str(Path(model_dir) / "model.safetensors"), device="cpu")
    with torch.device("meta"):
        model = IrisDiT(cfg.model)
    model.load_state_dict(weights, strict=True, assign=True)
    del weights
    model.to(device=device, dtype=torch.float32).requires_grad_(True)
    model.activation_checkpointing = "full"
    # Construct the frozen decoder on CPU. The trainer moves it to CUDA only
    # before the DiT backward pass, with all previous gradients cleared.
    encoder = Qwen3VLTextEncoder(cfg.text_encoder, device="cpu")
    cfg.text_encoder.pretrained = core.TEXT_REPO
    targets = [name for name, _ in model.named_parameters()]
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print(f"Iris本体: {sum(p.numel() for p in model.parameters()):,}パラメータを全て学習 / FP32保持・BF16計算")
    return model, encoder, cfg, targets


def storage_and_memory_estimate(model):
    """Tensor-size estimate, excluding activations, CUDA and transient casts."""
    weights = sum(t.numel() * t.element_size() for t in model.state_dict().values())
    gradients = sum(p.numel() * p.element_size() for p in model.parameters())
    optimizer = 0
    for parameter in model.parameters():
        n = parameter.numel()
        optimizer += n * 8 if n < 4096 else n * 2 + math.ceil(n / 256) * 8 + 2048
    checkpoint = weights + optimizer + 64 * 1024**2
    export = sum(t.numel() * (2 if t.is_floating_point() else t.element_size())
                 for t in model.state_dict().values())
    return dict(weight_gib=weights / 1024**3, gradient_gib=gradients / 1024**3,
        optimizer_estimated_gib=optimizer / 1024**3,
        persistent_training_tensor_gib=(weights + gradients + optimizer) / 1024**3,
        checkpoint_estimated_bytes=checkpoint, checkpoint_estimated_gib=checkpoint / 1024**3,
        bf16_export_estimated_gib=export / 1024**3,
        drive_peak_estimated_gib=(3 * checkpoint + export) / 1024**3,
        activations_included=False, measured_on_a100=False)


class FullFTTrainer(core.LoRATrainer):
    checkpoint_tensor_key = "model"
    gradient_clip = 0.5

    def __init__(self, model, encoder, cfg, targets, dataset, settings, run_dir, scratch, device="cuda"):
        settings = dict(settings)
        settings.setdefault("micro_batch_size", 1)
        self.micro_batch_size = settings["micro_batch_size"]
        self.bucket_order = BucketBatchOrder(dataset.records, self.micro_batch_size, settings["seed"])
        super().__init__(model, encoder, cfg, targets, dataset, settings, run_dir, scratch, device)

    def index_at(self, cursor):
        if self.micro_batch_size == 1:
            return super().index_at(cursor)
        epoch, offset = divmod(cursor, self.bucket_order.epoch_size)
        if epoch != self._epoch:
            self._order = self.bucket_order.order(epoch)
            self._epoch = epoch
        return self._order[offset]

    def _validate_trainable_model(self, parameters):
        if (not parameters or core.lora_modules(self.model) or
            any(not p.requires_grad or p.dtype != torch.float32 for p in self.model.parameters())):
            raise ValueError("全体FTはLoRAなし・Iris全パラメータをFP32で学習する構成です。")

    def _create_optimizer(self, parameters):
        if self.device.type == "cuda":
            from bitsandbytes.optim import AdamW8bit
            version = importlib.metadata.version("bitsandbytes")
            if version != BITSANDBYTES_VERSION:
                raise RuntimeError(f"bitsandbytes=={BITSANDBYTES_VERSION}で環境を準備してください。")
            self.optimizer_identity = f"bitsandbytes.AdamW8bit/{version}"
            return AdamW8bit(parameters, lr=self.settings["learning_rate"], betas=(0.9, 0.95),
                             weight_decay=0, min_8bit_size=4096)
        # Only the small CPU behavioral tests use this reference optimizer.
        # The Colab notebook requires a real A100 before reaching this code.
        self.optimizer_identity = "torch.AdamW/cpu-reference-tests"
        return torch.optim.AdamW(parameters, lr=self.settings["learning_rate"],
                                betas=(0.9, 0.95), weight_decay=0)

    def _build_recipe(self, targets):
        keys = ("seed", "learning_rate", "warmup_steps", "grad_accum", "text_dropout")
        recipe = dict(schema=core.SCHEMA_VERSION, training_mode="full-iris-fp32-v1",
            source_revision=core.SOURCE_REVISION, base_repo=core.BASE_REPO,
            base_revision=core.BASE_REVISION, text_repo=core.TEXT_REPO, text_revision=core.TEXT_REVISION,
            dataset_fingerprint=self.dataset.fingerprint, data=core.data_recipe(self.settings),
            training={key: self.settings[key] for key in keys}, targets=list(targets),
            model=asdict(self.cfg.model), flow=asdict(self.cfg.flow), base_dtype="torch.float32",
            compute_dtype="torch.bfloat16" if self.device.type == "cuda" else "torch.float32",
            optimizer=self.optimizer_identity, gradient_clip=self.gradient_clip, ema=False,
            conditioning="qwen-step-batch-cpu-offload-v1")
        if self.micro_batch_size > 1:
            recipe["training"]["micro_batch_size"] = self.micro_batch_size
            recipe["sampler"] = dict(version="seeded-bucket-tail-padding-v1",
                epoch_size=self.bucket_order.epoch_size, padding_per_epoch=self.bucket_order.padding)
        return recipe

    def _autocast(self):
        # Keeping every cast FP32 weight cached adds another ~6 GB. Checkpoint
        # recomputation recasts the active block instead, while updates stay FP32.
        return torch.autocast(self.device.type, dtype=torch.bfloat16,
                              enabled=self.device.type == "cuda", cache_enabled=False)

    def _build_null(self):
        from iris3b.text.base import TextEncoding
        self._conditionings = {}
        try:
            self.encoder.to(self.device)
            with torch.no_grad():
                encoding = self.encoder.null()
                return TextEncoding(encoding.embeddings.detach().cpu(), encoding.mask.detach().cpu())
        finally:
            self.encoder.to("cpu")
            if self.device.type == "cuda":
                torch.cuda.empty_cache()

    def _prepare_step(self, count=None):
        from iris3b.text.base import TextEncoding
        if any(p.grad is not None for p in self.model.parameters()):
            raise RuntimeError("QwenをGPUへ移す前に前回の勾配を解放してください。")
        count = self.settings["grad_accum"] * self.micro_batch_size if count is None else count
        self._conditionings.clear()
        captions = [self.dataset.records[self.index_at(self.cursor + offset)]["prompt"]
                    for offset in range(count)]
        try:
            self.encoder.to(self.device)
            with torch.no_grad():
                encoding = self.encoder.encode(captions)
                drops = torch.rand(count) < self.settings["text_dropout"]
                for offset in range(count):
                    if bool(drops[offset]):
                        embedding, mask = self.null.embeddings, self.null.mask
                    else:
                        embedding = encoding.embeddings[offset:offset + 1]
                        mask = encoding.mask[offset:offset + 1]
                    self._conditionings[self.cursor + offset] = TextEncoding(
                        embedding.detach().to(self.device), mask.detach().to(self.device))
        finally:
            # Qwen is absent from CUDA during *all* accumulation microsteps.
            self.encoder.to("cpu")
            if self.device.type == "cuda":
                torch.cuda.empty_cache()

    def _clear_step(self):
        self._conditionings.clear()
        self.encoder.to("cpu")

    def micro_loss(self, cursor):
        cursors = range(cursor, cursor + self.micro_batch_size)
        if any(position not in self._conditionings for position in cursors):
            raise RuntimeError("optimizerステップ前にテキスト条件を準備してください。")
        images = [self.dataset[self.index_at(position)][0] for position in cursors]
        if len({tuple(image.shape) for image in images}) != 1:
            raise RuntimeError("同じ縦横比バケットの画像をバッチにまとめてください。")
        image = torch.stack(images).to(self.device, dtype=torch.float32)
        embedding = torch.cat([self._conditionings[position].embeddings for position in cursors])
        mask = torch.cat([self._conditionings[position].mask for position in cursors])
        with self._autocast():
            return self.flow.training_loss(self.model, image, embedding,
                                           model_kwargs={"y_mask": mask}).loss

    def sanity_check(self):
        state = core.rng_state()
        self.optimizer.zero_grad(set_to_none=True)
        try:
            self._prepare_step(count=self.micro_batch_size)
            super().sanity_check()
        finally:
            self._clear_step()
            core.restore_rng(state)

    def _checkpoint_weights(self):
        return core.cpu_tree(self.model.state_dict())

    def _restore_weights(self, state):
        expected = self.model.state_dict()
        if (state.keys() != expected.keys() or
            any(state[name].shape != tensor.shape or state[name].dtype != tensor.dtype
                for name, tensor in expected.items())):
            raise ValueError("全体FTの保存モデルの名前・形状・dtypeが一致しません。")
        self.model.load_state_dict(state, strict=True)

    def _immutable_setting_keys(self):
        return ("seed", "learning_rate", "warmup_steps", "grad_accum", "micro_batch_size", "text_dropout",
                "sample_count", "resolution", "aspect_buckets", "min_aesthetic_tier", "sfw_only")

    def update_runtime_settings(self, settings):
        settings = dict(settings)
        settings.setdefault("micro_batch_size", 1)
        super().update_runtime_settings(settings)

    def save(self):
        if self.invalid_optimizer_state:
            return super().save()  # same invalid-state guard as LoRA
        if self.step == 0:
            print("学習更新が未完了のため全体FTチェックポイントは保存しません。", flush=True)
            return None
        if self.last_saved_step != self.step:
            needed = storage_and_memory_estimate(self.model)["checkpoint_estimated_bytes"]
            if shutil.disk_usage(self.scratch).free < needed:
                raise OSError("全体FTの途中保存用にColab側のディスク空きを確保してください。")
        return super().save()

    @torch.no_grad()
    def sample(self, prompts, steps=50, seed=42, compare_base=False):
        if compare_base:
            raise ValueError("全体FTのベース比較には、学習前のstep 0作例を使ってください。")
        try:
            self.encoder.to(self.device)
            return super().sample(prompts, steps=steps, seed=seed, compare_base=False)
        finally:
            self.encoder.to("cpu")
            if self.device.type == "cuda":
                torch.cuda.empty_cache()


def export_full_model(trainer, output_dir):
    """Export complete BF16 official Iris weights; keep FP32 resume files intact."""
    from omegaconf import OmegaConf
    from safetensors.torch import save_file
    if trainer.step == 0 or trainer.invalid_optimizer_state:
        raise RuntimeError("正常な学習済みモデルを読み込んでから書き出してください。")
    output_dir = Path(output_dir) / f"step-{trainer.step:08d}"
    staging = trainer.scratch / "full-export"
    staging.mkdir(parents=True, exist_ok=True)
    local_weights, local_config = staging / "model.safetensors", staging / "config.yaml"
    try:
        weights = {name: tensor.detach().cpu().to(torch.bfloat16 if tensor.is_floating_point() else tensor.dtype)
                   .contiguous().clone() for name, tensor in trainer.model.state_dict().items()}
        save_file(weights, str(local_weights))
        del weights
        text = asdict(trainer.cfg.text_encoder)
        text["pretrained"], text["null_embed_dir"] = core.TEXT_REPO, "output/pretrained_models"
        OmegaConf.save(OmegaConf.create(dict(model=asdict(trainer.cfg.model), text_encoder=text,
                                            flow=asdict(trainer.cfg.flow))), local_config)
        weights_hash = core.verified_copy(local_weights, output_dir / local_weights.name)
        config_hash = core.verified_copy(local_config, output_dir / local_config.name)
        core.atomic_json(output_dir / "export_manifest.json", dict(format="iris3b-full-bf16-v1",
            optimizer_step=trainer.step, signature=trainer.signature, recipe=trainer.recipe,
            files={"model.safetensors": weights_hash, "config.yaml": config_hash}))
    finally:
        local_weights.unlink(missing_ok=True)
        local_config.unlink(missing_ok=True)
    print(f"全体FTモデルを公式Iris形式で保存しました: {output_dir}")
    return output_dir


In [ ]:
%%writefile /content/iris_full_ft_phase_colab.py
"""Continue full FT on a new dataset while retaining all training state."""
from __future__ import annotations

import hashlib
import json
import os
from pathlib import Path

import torch

import iris_lora_colab as core
from iris_full_ft_colab import FullFTTrainer


def state_sample_digest(value):
    """Compare every tensor's shape/dtype and boundary samples without a full GPU copy."""
    digest = hashlib.sha256()

    def visit(item):
        if isinstance(item, torch.Tensor):
            digest.update(str((tuple(item.shape), str(item.dtype))).encode())
            flat = item.detach().reshape(-1)
            samples = torch.cat((flat[:16], flat[-16:])) if flat.numel() else flat
            digest.update(samples.cpu().contiguous().view(torch.uint8).numpy().tobytes())
        elif isinstance(item, dict):
            for key in sorted(item, key=str):
                digest.update(str(key).encode())
                visit(item[key])
        elif isinstance(item, (list, tuple)):
            for child in item:
                visit(child)
        else:
            digest.update(repr(item).encode())

    visit(value)
    return digest.hexdigest()


class PhaseFTTrainer(FullFTTrainer):
    """Global optimizer/cursor accounting, with a new dataset starting at local zero."""

    def __init__(self, *args, parent_record, learning_rate_transition=None, **kwargs):
        self.parent_record = dict(parent_record)
        self.learning_rate_transition = dict(learning_rate_transition) if learning_rate_transition else None
        if self.learning_rate_transition and (set(self.learning_rate_transition) != {"from", "to"}
                or not all(isinstance(v, (int, float)) and v > 0
                           for v in self.learning_rate_transition.values())):
            raise ValueError("An explicit positive from/to learning-rate transition is required.")
        if (not core.CHECKPOINT_NAME.fullmatch(self.parent_record["file"])
                or self.parent_record["step"] < 1
                or len(self.parent_record["sha256"]) != 64):
            raise ValueError("Invalid parent checkpoint identity.")
        self.phase_start_step = self.parent_record["step"]
        self.phase_start_cursor = self.parent_record["cursor"]
        super().__init__(*args, **kwargs)
        if (self.learning_rate_transition
                and self.learning_rate_transition["to"] != self.settings["learning_rate"]):
            raise ValueError("The target learning rate must match the continuation settings.")
        effective_batch = self.settings["grad_accum"] * self.micro_batch_size
        if self.phase_start_cursor != self.phase_start_step * effective_batch:
            raise ValueError("Parent effective batch must match the continuation.")
        if any(row["aesthetic_tier"] < self.settings["min_aesthetic_tier"]
               for row in self.dataset.records):
            raise ValueError("Continuation images must meet the configured quality threshold.")

    def _build_recipe(self, targets):
        recipe = super()._build_recipe(targets)
        recipe["phase"] = dict(version="retain-state-new-dataset-v1",
            parent_checkpoint=self.parent_record, start_data_cursor=0)
        if self.learning_rate_transition:
            recipe["phase"].update(version="retain-state-new-dataset-lr-transition-v2",
                                  learning_rate_transition=self.learning_rate_transition)
        return recipe

    def index_at(self, cursor):
        local_cursor = cursor - self.phase_start_cursor
        if local_cursor < 0:
            raise ValueError("Load the parent checkpoint before drawing the new dataset.")
        return super().index_at(local_cursor)

    def start_from_parent(self, parent_run_dir):
        if (self.run_dir / "checkpoints/index.json").is_file():
            raise RuntimeError("The continuation already has a checkpoint; call resume().")
        parent_run_dir = Path(parent_run_dir)
        if parent_run_dir.resolve() == self.run_dir.resolve():
            raise ValueError("The parent and continuation must have separate save directories.")
        index = json.loads((parent_run_dir / "checkpoints/index.json").read_text())
        if index.get("schema") != core.SCHEMA_VERSION or self.parent_record not in index["checkpoints"]:
            raise RuntimeError("The exact verified parent checkpoint is absent.")
        source = parent_run_dir / "checkpoints" / self.parent_record["file"]
        local = self.scratch / ("phase-parent-" + self.parent_record["file"])
        core.verified_copy(source, local, self.parent_record["sha256"])
        try:
            payload = torch.load(local, map_location="cpu", weights_only=True, mmap=os.name != "nt")
            if (payload.get("schema") != core.SCHEMA_VERSION
                    or payload.get("step") != self.phase_start_step
                    or payload.get("cursor") != self.phase_start_cursor
                    or payload.get("signature") != core.stable_hash(payload["recipe"])):
                raise RuntimeError("Invalid parent schema, position, or recipe signature.")
            # Only data selection and its sampler length may change in this phase.
            ignore = {"data", "dataset_fingerprint", "sampler", "phase"}
            old = {k: v for k, v in payload["recipe"].items() if k not in ignore}
            current = {k: v for k, v in self.recipe.items() if k not in ignore}
            if self.learning_rate_transition:
                old = dict(old, training=dict(old["training"]))
                if old["training"]["learning_rate"] != self.learning_rate_transition["from"]:
                    raise ValueError("The parent learning rate does not match the authorized transition.")
                old["training"]["learning_rate"] = self.learning_rate_transition["to"]
            if old != current:
                raise ValueError("Architecture, optimizer, schedule, and training conditions must match the parent.")
            old_data = payload["recipe"]["data"]
            new_data = self.recipe["data"]
            data_ignore = {"sample_count", "min_aesthetic_tier", "selection", "exclusion",
                           "repo", "revision", "columns", "parquet_glob"}
            if ({k: v for k, v in old_data.items() if k not in data_ignore}
                    != {k: v for k, v in new_data.items() if k not in data_ignore}):
                raise ValueError("Resolution, preprocessing, seed and source revision must remain unchanged.")
            model_digest = state_sample_digest(payload["model"])
            optimizer_digest = state_sample_digest(payload["optimizer"])
            scheduler_state = payload["scheduler"]
            self._restore_weights(payload["model"])
            self.optimizer.load_state_dict(payload["optimizer"])
            self.scheduler.load_state_dict(scheduler_state)
            self.step, self.cursor = payload["step"], payload["cursor"]
            self.last_loss, self.committed_rng = payload["last_loss"], payload["rng"]
            self._epoch, self._order = None, None
            self.invalid_optimizer_state = False
            if state_sample_digest(self.model.state_dict()) != model_digest:
                raise RuntimeError("The parent model was not restored correctly.")
            if state_sample_digest(self.optimizer.state_dict()) != optimizer_digest:
                raise RuntimeError("The parent optimizer was not restored correctly.")
            if self.scheduler.state_dict() != scheduler_state:
                raise RuntimeError("The parent scheduler was not restored correctly.")
            moments_digest = state_sample_digest(self.optimizer.state_dict()["state"])
            if self.learning_rate_transition:
                if any(group["lr"] != self.learning_rate_transition["from"]
                       for group in self.optimizer.param_groups):
                    raise ValueError("The saved effective learning rate differs from the transition source.")
                new_lr = self.learning_rate_transition["to"]
                for group in self.optimizer.param_groups:
                    group["lr"] = group["initial_lr"] = new_lr
                updated_schedule = dict(scheduler_state,
                    base_lrs=[new_lr] * len(self.optimizer.param_groups),
                    _last_lr=[new_lr] * len(self.optimizer.param_groups))
                self.scheduler.load_state_dict(updated_schedule)
                if state_sample_digest(self.optimizer.state_dict()["state"]) != moments_digest:
                    raise RuntimeError("Changing the learning rate changed the optimizer moments.")
            core.restore_rng(self.committed_rng)
            if state_sample_digest(core.rng_state()) != state_sample_digest(self.committed_rng):
                raise RuntimeError("The parent random state was not restored correctly.")
            receipt = dict(parent=self.parent_record, retained_step=self.step,
                retained_global_cursor=self.cursor, new_dataset_cursor=0,
                parent_dataset_fingerprint=payload["recipe"]["dataset_fingerprint"],
                dataset_fingerprint=self.dataset.fingerprint,
                model_samples_verified=True, optimizer_samples_verified=True,
                scheduler_verified=True, rng_verified=True,
                model_sample_digest=model_digest, optimizer_sample_digest=optimizer_digest,
                learning_rate=self.optimizer.param_groups[0]["lr"],
                target_optimizer_step=self.settings["max_steps"])
            if self.learning_rate_transition:
                receipt.update(learning_rate_transition=self.learning_rate_transition,
                    optimizer_moments_retained=True,
                    optimizer_moments_digest=moments_digest,
                    retained_scheduler_epoch=self.scheduler.state_dict()["last_epoch"],
                    scheduler_base_lrs=self.scheduler.state_dict()["base_lrs"])
            core.atomic_json(self.run_dir / "phase_start.json", receipt)
        finally:
            local.unlink(missing_ok=True)
        print(f"Parent restored: step {self.step}, model + optimizer + scheduler + RNG. New dataset starts at 0.", flush=True)
        return receipt

    def save(self):
        if self.step <= self.phase_start_step:
            return None
        return super().save()


In [ ]:
%%writefile /content/iris_unseen_colab.py
"""Build a signed exclusion ledger from verified earlier image pools."""
from __future__ import annotations

import iris_lora_colab as core


def build_exclusion_ledger(manifests):
    if not manifests:
        raise ValueError("Previous training manifests are required.")
    identifiers, image_hashes, fingerprints = set(), set(), []
    for manifest in manifests:
        expected = core.stable_hash({"recipe": manifest["recipe"], "records": manifest["records"]})
        if expected != manifest["fingerprint"]:
            raise ValueError("A previous dataset manifest failed its fingerprint check.")
        fingerprints.append(expected)
        for row in manifest["records"]:
            identifiers.add(str(row["image_id"]))
            image_hashes.add(row["image_sha256"])
    return dict(schema=1, prior_dataset_fingerprints=sorted(fingerprints),
                image_ids=sorted(identifiers), image_sha256=sorted(image_hashes))


def exclusion_recipe(ledger):
    return dict(sha256=core.stable_hash(ledger), excluded_id_count=len(ledger["image_ids"]),
                excluded_image_hash_count=len(ledger["image_sha256"]),
                prior_dataset_fingerprints=ledger["prior_dataset_fingerprints"])


In [ ]:
%%writefile /content/iris_preview_colab.py
"""A configurable one-time preview without changing the training trajectory."""
from __future__ import annotations

import copy
import gc
import time
from pathlib import Path

import torch
from PIL import Image, ImageDraw

import iris_lora_colab as core
from iris_full_ft_phase_colab import state_sample_digest

DEFAULT_PROMPTS = [
    dict(name="sample", prompt="An illustration of a flower in a pot.", seed=42, width=1024, height=1024),
]


def preview_once(trainer, run_dir, scratch, prompts, *, target_step=300, generate_fn=None):
    """Save once at target_step; preserve RNG, optimizer, weights and mode."""
    if not isinstance(target_step, int) or target_step < 1:
        raise ValueError("target_step must be a positive integer.")
    if trainer.step != target_step:
        return None
    run_dir, scratch = Path(run_dir), Path(scratch)
    output_dir = run_dir / f"previews/step-{target_step:08d}"
    local_dir = scratch / f"previews/step-{target_step:08d}"
    shift = float(getattr(trainer.cfg.flow, "shift", 4.0))
    policy = dict(optimizer_step=target_step,
                  added_optimizer_steps=target_step - getattr(trainer, "phase_start_step", 0), prompts=prompts,
                  sampling_steps=50, cfg_scale=3.0, shift=shift)
    manifest_path = output_dir / "manifest.json"
    if manifest_path.is_file():
        import json
        saved = json.loads(manifest_path.read_text())
        if all(saved.get(key) == value for key, value in policy.items()):
            if all(core.file_hash(run_dir / item["file"]) == item["sha256"]
                   for item in saved["images"]):
                return saved
        raise RuntimeError("The existing preview has a different identity or hash.")
    if generate_fn is None:
        from iris3b.sampling import generate
        generate_fn = generate
    output_dir.mkdir(parents=True, exist_ok=True)
    local_dir.mkdir(parents=True, exist_ok=True)
    rng = core.rng_state()
    rng_digest = state_sample_digest(rng)
    model_digest = state_sample_digest(trainer.model.state_dict())
    optimizer_digest = state_sample_digest(trainer.optimizer.state_dict())
    scheduler = copy.deepcopy(trainer.scheduler.state_dict())
    was_training = trainer.model.training
    stats = {name: getattr(trainer.encoder, name) for name in
             ("calls", "overflow_rows", "overflow_tokens", "max_caption_tokens")
             if hasattr(trainer.encoder, name)}
    images, contact_sheet, error = [], None, None
    started = time.monotonic()
    try:
        assert not any(parameter.grad is not None for parameter in trainer.model.parameters())
        trainer.model.eval()
        trainer.encoder.to(trainer.device)
        for item in prompts:
            with torch.no_grad(), trainer._autocast():
                output = generate_fn(trainer.model, trainer.encoder, [item["prompt"]],
                    height=item["height"], width=item["width"], steps=50,
                    cfg_scale=3.0, shift=shift,
                    generator=torch.Generator(device=trainer.device).manual_seed(item["seed"]),
                    device=trainer.device, num_train_timesteps=trainer.cfg.flow.num_train_timesteps,
                    prediction=trainer.cfg.flow.prediction)[0]
            assert output.shape == (3, item["height"], item["width"])
            assert torch.isfinite(output).all().item()
            array = output.detach().float().cpu().add(1).mul(127.5).clamp(0, 255)
            array = array.permute(1, 2, 0).round().to(torch.uint8).numpy()
            assert float(array.std()) > 1
            local_image = local_dir / (item["name"] + ".png")
            Image.fromarray(array).save(local_image)
            target = output_dir / local_image.name
            checksum = core.verified_copy(local_image, target)
            images.append(dict(file=str(target.relative_to(run_dir)), sha256=checksum,
                optimizer_step=target_step, added_optimizer_steps=policy["added_optimizer_steps"], **item,
                sampling_steps=50, cfg_scale=3.0, shift=shift, pixel_std=float(array.std())))
            del output, array
            print(f"STEP_{target_step}_PREVIEW: {len(images)}/{len(prompts)} / {item['name']}", flush=True)
        sheet = Image.new("RGB", (1024, ((len(images) + 1) // 2) * 548), "white")
        draw = ImageDraw.Draw(sheet)
        for index, item in enumerate(images):
            x, y = (index % 2) * 512, (index // 2) * 548
            draw.text((x + 8, y + 8), f"Step {target_step} / +{policy['added_optimizer_steps']} / {item['name']}", fill="black")
            with Image.open(run_dir / item["file"]) as source:
                source.thumbnail((512, 512))
                sheet.paste(source, (x + (512 - source.width) // 2, y + 30))
        local_sheet = local_dir / "contact-sheet.png"
        sheet.save(local_sheet)
        target = output_dir / local_sheet.name
        checksum = core.verified_copy(local_sheet, target)
        contact_sheet = dict(file=str(target.relative_to(run_dir)), sha256=checksum)
    except Exception as exc:
        error = dict(type=type(exc).__name__, message=str(exc))
        print(f"STEP_{target_step}_PREVIEW_FAILED: {error}; preserving training state and continuing.", flush=True)
    finally:
        try:
            trainer.encoder.to("cpu")
        finally:
            trainer.model.train(was_training)
            for name, value in stats.items():
                setattr(trainer.encoder, name, value)
            core.restore_rng(rng)
            gc.collect()
            if trainer.device.type == "cuda":
                torch.cuda.empty_cache()
    assert state_sample_digest(core.rng_state()) == rng_digest
    assert state_sample_digest(trainer.model.state_dict()) == model_digest
    assert state_sample_digest(trainer.optimizer.state_dict()) == optimizer_digest
    assert trainer.scheduler.state_dict() == scheduler
    receipt = dict(policy, status="preview_complete" if error is None else "preview_failed",
        images=images, contact_sheet=contact_sheet, error=error,
        training_rng_restored=True, model_samples_verified=True,
        optimizer_samples_verified=True, scheduler_verified=True,
        seconds=time.monotonic() - started)
    core.atomic_json(manifest_path, receipt)
    print(f"STEP_{target_step}_PREVIEW_SAVED: {len(images)} images; training state restored.", flush=True)
    return receipt


In [ ]:
#@title 全体FTの設定
DATASET_REPO = "" #@param {type:"string"}
DATASET_REVISION = "" #@param {type:"string"}
PARQUET_GLOB = "*.parquet" #@param {type:"string"}
ID_COLUMN = "id" #@param {type:"string"}
IMAGE_COLUMN = "image" #@param {type:"string"}
PROMPT_COLUMN = "prompt" #@param {type:"string"}
QUALITY_COLUMN = "" #@param {type:"string"}
PARENT_RUN_NAME = "" #@param {type:"string"}
PARENT_LEARNING_RATE = 0.00001 #@param {type:"number"}
EXCLUSION_MANIFESTS_JSON = "[]" #@param {type:"string"}
RUN_NAME = "iris3b-fullft-continuation" #@param {type:"string"}
SAMPLE_COUNT = 5000 #@param {type:"integer"}
SEED = 42 #@param {type:"integer"}
RESOLUTION = 1024 #@param [512, 1024] {type:"raw"}
MIN_AESTHETIC_TIER = 0 #@param {type:"integer"}
ASPECT_BUCKETS = True #@param {type:"boolean"}
SFW_ONLY = False #@param {type:"boolean"}
LEARNING_RATE = 0.00001 #@param {type:"number"}
MICRO_BATCH_SIZE = 14 #@param {type:"integer"}
GRAD_ACCUM = 2 #@param {type:"integer"}
WARMUP_STEPS = 30 #@param {type:"integer"}
MAX_STEPS = 600 #@param {type:"integer"}
SESSION_MINUTES = 180 #@param {type:"integer"}
SAVE_EVERY = 100 #@param {type:"integer"}
SAVE_MINUTES = 30 #@param {type:"integer"}

import gc
import importlib
import json
import re
import shutil
import torch
from google.colab import drive
sys.path.insert(0, "/content")
import iris_lora_colab as iris
import iris_full_ft_colab as fullft
iris = importlib.reload(iris)
fullft = importlib.reload(fullft)

if (not re.fullmatch(r"[A-Za-z0-9_-]{1,80}", PARENT_RUN_NAME) or PARENT_RUN_NAME == RUN_NAME):
    raise ValueError("親のPARENT_RUN_NAMEを入力し、RUN_NAMEには別の名前を指定してください。")
if PARENT_LEARNING_RATE <= 0:
    raise ValueError("PARENT_LEARNING_RATEは正の値にしてください。")
if not re.fullmatch(r"[A-Za-z0-9_-]{1,80}", RUN_NAME):
    raise ValueError("RUN_NAMEは英数字・ハイフン・アンダースコアの1〜80文字にしてください。")
if min(SAMPLE_COUNT, MICRO_BATCH_SIZE, GRAD_ACCUM, MAX_STEPS, SESSION_MINUTES, SAVE_EVERY, SAVE_MINUTES) < 1:
    raise ValueError("枚数、ステップ数、時間、保存間隔は1以上にしてください。")
if not 0 <= SEED < 2**32 or WARMUP_STEPS < 0 or LEARNING_RATE <= 0:
    raise ValueError("seed、warmup、学習率を確認してください。")
dataset_settings = dict(dataset_repo=DATASET_REPO, dataset_revision=DATASET_REVISION,
    parquet_glob=PARQUET_GLOB, dataset_columns=dict(image_id=ID_COLUMN, image=IMAGE_COLUMN,
    prompt=PROMPT_COLUMN, aesthetic_tier=QUALITY_COLUMN or None))
iris.dataset_config(dict(dataset_settings, min_aesthetic_tier=MIN_AESTHETIC_TIER), require_source=True)
iris.flow_shift_for_resolution(RESOLUTION)
iris.check_a100()
if MICRO_BATCH_SIZE > 1 and torch.cuda.get_device_properties(0).total_memory < 75 * 1024**3:
    raise RuntimeError("この実バッチ設定はA100 80GB用です。40GBではMICRO_BATCH_SIZE=1から確認してください。")
memory = {line.split(":", 1)[0]: int(line.split()[1]) * 1024
          for line in Path("/proc/meminfo").read_text().splitlines() if ":" in line}
if memory["MemTotal"] < 32 * 1024**3:
    raise RuntimeError("全体FTの保存用に高RAM（32GiB以上）のColabランタイムを選んでください。")

if not Path("/content/drive/MyDrive").is_dir():
    drive.mount("/content/drive", timeout_ms=600000)
DRIVE_ROOT = Path("/content/drive/MyDrive/Iris3B_FullFT")
RUN_DIR = DRIVE_ROOT / RUN_NAME
SCRATCH = Path("/content/iris-fullft-work") / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
SCRATCH.mkdir(parents=True, exist_ok=True)
if shutil.disk_usage("/content").free < 60 * 1024**3 and "trainer" not in globals():
    raise RuntimeError("モデル取得・大きな途中保存用にColab側の空き60GiB以上を用意してください。")
settings = dict(**dataset_settings, sample_count=SAMPLE_COUNT, seed=SEED, resolution=RESOLUTION,
    aspect_buckets=ASPECT_BUCKETS, min_aesthetic_tier=MIN_AESTHETIC_TIER, sfw_only=SFW_ONLY,
    learning_rate=LEARNING_RATE, micro_batch_size=MICRO_BATCH_SIZE, grad_accum=GRAD_ACCUM, warmup_steps=WARMUP_STEPS,
    text_dropout=0.1, max_steps=MAX_STEPS, session_minutes=SESSION_MINUTES,
    save_every=SAVE_EVERY, save_minutes=SAVE_MINUTES, keep_checkpoints=2)
print("保存先:", RUN_DIR)
print("全体FT /", RESOLUTION, "px / 実バッチ", MICRO_BATCH_SIZE, "/ 累積", GRAD_ACCUM,
      "/ 実効バッチ", MICRO_BATCH_SIZE * GRAD_ACCUM, "/ 学習率", LEARNING_RATE)
print("Google Driveの空き目安: 約200GB（データ、複数世代の保存、最終モデル、Driveのごみ箱を含む）")


In [ ]:
#@title 親チェックポイントと除外条件を確認
import iris_full_ft_phase_colab as phaseft
import iris_unseen_colab as unseen
phaseft = importlib.reload(phaseft)
unseen = importlib.reload(unseen)
PARENT_DIR = DRIVE_ROOT / PARENT_RUN_NAME
parent_index = json.loads((PARENT_DIR / "checkpoints/index.json").read_text(encoding="utf-8"))
if parent_index.get("schema") != iris.SCHEMA_VERSION or not parent_index.get("checkpoints"):
    raise RuntimeError("この公開版の正常な親チェックポイントが必要です。")
phase_receipt = RUN_DIR / "phase_start.json"
if phase_receipt.is_file():
    PARENT_RECORD = json.loads(phase_receipt.read_text(encoding="utf-8"))["parent"]
else:
    PARENT_RECORD = max(parent_index["checkpoints"], key=lambda item: item["step"])
if PARENT_RECORD not in parent_index["checkpoints"]:
    raise RuntimeError("最初に使った親チェックポイントを保存したまま再開してください。")
if MAX_STEPS <= PARENT_RECORD["step"]:
    raise ValueError("MAX_STEPSには親のstepより大きい通算目標を指定してください。")
paths = json.loads(EXCLUSION_MANIFESTS_JSON)
if not isinstance(paths, list) or any(not isinstance(path, str) for path in paths):
    raise ValueError("EXCLUSION_MANIFESTS_JSONはmanifest相対パスのJSON配列です。")
manifests = []
for relative in paths:
    path = (DRIVE_ROOT / relative).resolve()
    if not path.is_relative_to((DRIVE_ROOT / "datasets").resolve()) or path.name != "manifest.json":
        raise ValueError("datasets/配下のmanifest.jsonを指定してください。")
    manifests.append(json.loads(path.read_text(encoding="utf-8")))
ledger = unseen.build_exclusion_ledger(manifests) if manifests else None
if ledger is not None:
    settings["exclusion"] = unseen.exclusion_recipe(ledger)
transition = ({"from": PARENT_LEARNING_RATE, "to": LEARNING_RATE}
              if LEARNING_RATE != PARENT_LEARNING_RATE else None)
print("親step:", PARENT_RECORD["step"], "/ 通算目標:", MAX_STEPS)


In [ ]:
#@title 新しいデータを準備または復元
dataset = iris.prepare_dataset(DRIVE_ROOT, SCRATCH, settings, exclusion=ledger)
print("画像数:", len(dataset), "/ データ識別子:", dataset.fingerprint[:16])


In [ ]:
#@title 全体FTモデルを読み込み、親状態または途中保存を復元
model, encoder, cfg, targets = fullft.load_full_models(settings, Path("/content/iris-model-cache"))
trainer = phaseft.PhaseFTTrainer(model, encoder, cfg, targets, dataset, settings, RUN_DIR, SCRATCH,
    parent_record=PARENT_RECORD, learning_rate_transition=transition)
if (RUN_DIR / "checkpoints/index.json").is_file():
    trainer.resume()
else:
    trainer.start_from_parent(PARENT_DIR)
print("復元済みstep:", trainer.step, "/ 通算画像位置:", trainer.cursor)


In [ ]:
#@title 目標または時間上限まで学習して保存
trainer.update_runtime_settings(settings)
trainer.train(max_steps=MAX_STEPS)
print("保存済みstep:", trainer.step)


In [ ]:
#@title 公式形式のBF16モデルを書き出す
export_dir = fullft.export_full_model(trainer, RUN_DIR / "export" / f"step-{trainer.step:08d}")
print("モデルとconfigの保存先:", export_dir)


In [ ]:
#@title Driveへの送信を完了し、切断できる状態にする
from google.colab import drive
print("Google Driveへの未送信データを同期しています。", flush=True)
drive.flush_and_unmount(timeout_ms=1800000)
print("Driveへの送信完了。GPUランタイムを接続解除できます。", flush=True)
